# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors: Data Exploration with `mlcroissant`

This notebook demonstrates how to explore and analyze the FAIR\u00b2 dataset ("Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution") using the [mlcroissant](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant --quiet

## 1. Data Loading

We load metadata and records from the FAIR\u00b2 dataset using the `mlcroissant` library. The Croissant schema URL for this dataset is:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)

# Accessing metadata as a single object
meta = dataset.metadata

print(f"{meta.name}: {meta.description}")

## 2. Data Overview

Review available record sets ("tables") in the dataset. For each record set, we'll display the fields (columns) available and their stable Croissant `@id`s for reliable downstream references.
**Note**: All identifiers (`@id`) are referenced as required for reproducibility.

In [ ]:
# List all record sets (by their @id) and their fields

record_sets = dataset.record_sets
print("Available record sets:")
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, dataType: {field.data_type})")
    print()

# We'll pick the first record set for detailed inspection below,
# but you may choose another by using its @id.

## 3. Data Extraction

We'll load the primary data table into a pandas DataFrame, referencing the RecordSet and field `@id`s identified above.

*The following cell loads all record sets into DataFrames by their `@id`s. Adjust the list if you wish to load only a subset or a particular record set.*

In [ ]:
# Prepare a mapping from record set @id to DataFrame

dataframes = {}
all_record_set_ids = [rs.id for rs in dataset.record_sets]

# Example: load all record sets
for record_set_id in all_record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded RecordSet {record_set_id} with shape: {df.shape}")

# For demonstration, select the first RecordSet as the primary data table
main_record_set_id = all_record_set_ids[0]
print(f"\nFields in RecordSet {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())

# Show first few rows
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Now, let's explore and process the main table. We'll pick a numeric field using its `@id` (see the field listing above), apply some filters and normalization, and group/category as an initial analysis step.

> **Please identify a numeric field and a grouping (categorical) field from cell 5 above (column names are the field `@id`s). For demonstration, we will use the first numeric field we find and the first string/categorical field present. You can change these below.**

In [ ]:
# Find a numeric field from the DataFrame's columns by type (example: floats or ints)
df = dataframes[main_record_set_id]

numeric_field_id = None
group_field_id = None

# Attempt to find a numeric field (float, int) for demo
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break

# Attempt to find a non-numeric (string/categorical) field for grouping
for col in df.columns:
    if pd.api.types.is_string_dtype(df[col]):
        group_field_id = col
        break

print(f"Using numeric field: {numeric_field_id}")
print(f"Using grouping field: {group_field_id}\n")

# If no numeric field found, skip further EDA
if numeric_field_id is None:
    print("No numeric field found for demonstration.")
else:
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[numeric_field_id + '_normalized'] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

    # Grouping
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped average of {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization

Let's visualize the distribution of the numeric field and its relationship with the grouping field (if identified above).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    # Histogram of the numeric field
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=10)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot of numeric field by group (if grouping field exists)
    if group_field_id in df.columns:
        plt.figure(figsize=(10,4))
        order = df[group_field_id].value_counts().index[:10] # Only top 10 categories
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df, order=order)
        plt.title(f"Distribution of {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

In this notebook, we've demonstrated:
- How to load and inspect a Croissant-formatted biomedical dataset using `mlcroissant`.
- Extraction of record sets and fields using their stable `@id`s.
- Simple exploratory data analysis and normalization using pandas.
- Basic data visualization with Matplotlib/Seaborn.

Please adjust record set IDs, field IDs, and analysis as needed for your exploration and research questions!